# FloodSense Test Notebook

### Read Me First

Before you can use this notebook, you must configure your python environment.

*1. Create virtual environment*

- In your terminal, navigate to your project folder. Alternatively you can type in your windows address bar **cmd** and this will open up a new terminal in your project directory.

- From your terminal enter the following command:

```python
python -m venv floodSense
```

- This creates your virtual environment in your project folder. Then navigate into your newly created environment and then activate it by entering the following command:

```python
\Scripts\activate
```

*2. Install floodsense*

- While still in the newly created environment, install the floodsense package by entering the following command. Note that this command should work if you have git installed on your machine. If an error shows, then you need to install it. See here for more: https://git-scm.com/book/en/v2/Getting-Started-Installing-Git

```python
pip install git+https://github.com/charlieikosi/floodsense.git@main
```

*3. Install jupyter kernel*

- While still in the newly created environment, install jupyter kernel:

```python
pip install ipykernel
python -m ipykernel install --user --name=floodsense
```

*4. Refresh Jupyter Notebook*

- Once step 3 is complete, refresh this jupyter notebook or click on **Kernel** and select **Restart Kernel...**
- Once restared, click **Kernel** once more and go to **Change Kernel...** A dialogue box pops up, click in the drop down and look for the virtual environment you just created called **floodsense**


*5. Run Workflow*

- If all goes smoothly, you are ready to run the workflow below

### Workflow

Ensure you have a a python environment configured before running the workflow. If you have not set one up, please follow the **Read Me First** instructions above.


In [ ]:
pip show floodsense

In [ ]:
# Import floodsense package

import floodsense
from floodsense import *
from floodsense.persistence import (
    find_previous_geojson,
    load_previous_polygons,
    update_persistence
)

In [ ]:
# Flood Detection Workflow

import os
import pystac_client
import planetary_computer
import geopandas as gpd

# --- CONFIGURATION ---
AOI = r"C:\Users\Charlie.Ikosi\OneDrive - PDP\Documents\floodsense\test\roi\AOI.shp" # Canterbury
#AOI = r"C:\Users\Charlie.Ikosi\OneDrive - PDP\Documents\floodsense\test\roi\Whitianga.shp" # Whitianga
#AOI = r"C:\Users\Charlie.Ikosi\OneDrive - PDP\Documents\floodsense\test\roi\NRC.shp" # NRC
#AOI = r"C:\Users\Charlie.Ikosi\OneDrive - PDP\Documents\floodsense\test\roi\otagoAOI.shp" # Otago
#AOI = r"C:\Users\Charlie.Ikosi\OneDrive - PDP\Documents\floodsense\test\roi\otagoAOI2.shp" # Otago2
#AOI = r"C:\Users\Charlie.Ikosi\OneDrive - PDP\Documents\floodsense\test\roi\Aoraki_Snow_Test.shp" # Aoraki Snow Test

BASELINE_SCENE_ID = "S1A_IW_GRDH_1SDV_20250419T073047_20250419T073116_058822_rtc" # Canterbury
#BASELINE_SCENE_ID = "S1C_IW_GRDH_1SDV_20260405T070704_20260405T070729_007077_00E551_rtc" # Whitianga
#BASELINE_SCENE_ID = "S1C_IW_GRDH_1SDV_20260504T071545_20260504T071613_007500_00F3A5_rtc" # NRC
#BASELINE_SCENE_ID = "S1D_IW_GRDH_1SDV_20260618T074531_20260618T074601_003287_005C13_rtc" # Otago
#BASELINE_SCENE_ID = "S1C_IW_GRDH_1SDV_20260131T073805_20260131T073830_006144_00C549_rtc" # Aoraki Snow Test

# Set the target range to start the day AFTER your baseline
TARGET_DATE_RANGE = "2025-04-20/2025-05-08" # Canterbury
#TARGET_DATE_RANGE = "2026-04-01/2026-04-19" # Whitianga
#TARGET_DATE_RANGE = "2026-07-01/2026-07-31" # Whitianga
#TARGET_DATE_RANGE = "2026-05-01/2026-05-28" # NRC
#TARGET_DATE_RANGE = "2026-07-06/2026-07-09" # Otago
#TARGET_DATE_RANGE = "2026-06-01/2026-07-31" # Aoraki Snow Test

ORBIT_STATE = "ascending"            
THRESHOLD_VAL = -2.5                   
SMOOTHING_WINDOW = 17                 
OUTPUT_DIR = "./output_payloads"

os.makedirs(OUTPUT_DIR, exist_ok=True)

aoi_gdf = gpd.read_file(AOI)
grid_id = str(
    aoi_gdf.iloc[0]["gridID"]
)
print(f"Grid ID: {grid_id}")

# === 1. FETCH & LOCK THE BASELINE ===
print(f"\n=== FETCHING BASELINE SCENE ===")
print(f"ID: {BASELINE_SCENE_ID}")

# Open the catalog directly to search by ID
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace
)

baseline_search = catalog.search(
    collections=["sentinel-1-rtc"],
    ids=[BASELINE_SCENE_ID]
)
baseline_items = baseline_search.item_collection()

if len(baseline_items) == 0:
    raise ValueError("Baseline scene not found! Check the ID and catalog connection.")

baseline_scene = baseline_items[0]

# Process the baseline into memory (Do this only once!)
print("Smoothing and formatting baseline data...")
data_baseline = load_and_crop_dual_pol(baseline_scene, AOI)
data_baseline = apply_spatial_tuning(data_baseline, window_size=SMOOTHING_WINDOW)


# === 2. FETCH TARGET SCENES ===
print(f"\n=== FETCHING TARGET SCENES FOR {TARGET_DATE_RANGE} ===")
target_items, my_aoi = get_rtc_catalog_items(AOI, TARGET_DATE_RANGE)
target_items = select_scenes_by_orbit(target_items, orbit_state=ORBIT_STATE)

if not target_items:
    print("No new scenes found in the target date range.")
else:
    # Sort chronologically (Oldest to Newest)
    target_items = sorted(target_items, key=lambda x: x.datetime)

    # === 3. CONTINUOUS MONITORING LOOP ===
    for current_scene in target_items:
        current_date_str = current_scene.datetime.strftime("%Y-%m-%d")
        print(f"\n--- PROCESSING TARGET DATE: {current_date_str} ---")
        
        try:
            # Load the current target scene
            data_current = load_and_crop_dual_pol(current_scene, AOI)
            data_current = apply_spatial_tuning(data_current, window_size=SMOOTHING_WINDOW)

            # Generate VV/VH change rasters
            change_vv, change_vh = calculate_dual_pol_change_db(
                data_baseline,
                data_current
            )
            
            # Apply the analytical pipeline
            wind_aware_flood_mask = calculate_wind_aware_mask(
                data_baseline, data_current, threshold_db=THRESHOLD_VAL
            )
            cleaned_flood_mask = apply_binary_median_filter(
                wind_aware_flood_mask, window_size=3
            )
            terrain_corrected_mask, dem, terrain_slope = (
                apply_terrain_mask(
                    cleaned_flood_mask,
                    my_aoi,
                    max_slope_degrees=5.0
                )
            )
            
            # --- NEW STEP: Apply the Permanent Water Mask ---
            final_flood_mask = apply_permanent_water_mask(
                terrain_corrected_mask, my_aoi
            )

            # Dynamic Export Naming
            raster_name = (
                f"{OUTPUT_DIR}/"
                f"{grid_id}_"
                f"flood_ext_"
                f"{current_date_str}.tif"
            )
            
            vector_name = (
                f"{OUTPUT_DIR}/"
                f"{grid_id}_"
                f"flood_ext_"
                f"{current_date_str}.geojson"
            )

            # Export the final masked version
            export_to_geotiff(final_flood_mask, raster_name)
            
            current_gdf = export_mask_to_polygons(
                final_flood_mask,
                output_filename=vector_name,
                event_date=current_date_str,
                event_datetime=current_scene.datetime.isoformat(),
                scene_id=current_scene.id,
                orbit_state=ORBIT_STATE,
                baseline_scene_id=baseline_scene.id,
                threshold_db=THRESHOLD_VAL,
                smoothing_window=SMOOTHING_WINDOW,
                processing_version="0.0.2",
                change_vv=change_vv,
                change_vh=change_vh,
                dem=dem,
                terrain_slope=terrain_slope
            )

            previous_file = find_previous_geojson(
                current_date_str,
                OUTPUT_DIR,
                grid_id
            )

            if previous_file is not None:

                print(
                    f"Found previous flood layer: "
                    f"{os.path.basename(previous_file)}"
                )
            
                previous_gdf = load_previous_polygons(
                    previous_file
                )
            
                current_gdf = update_persistence(
                    current_gdf,
                    previous_gdf
                )
            
                # Overwrite GeoJSON with persistence updates
                current_gdf.to_file(
                    vector_name,
                    driver="GeoJSON"
                )
            
                print(
                    "Persistence attributes updated."
                )
            
            else:
            
                print(
                    "No previous flood layer found. "
                    "Initializing persistence."
                )
            
        except Exception as e:
            print(f"Failed processing {current_date_str}: {e}")
            continue # Ensure the loop doesn't die if one scene has a corrupted STAC link

### Baseline Scene Selection

Use this to manually choose the baseline scene

In [ ]:
# Identify Baseline Scene
AOI = r"C:\Users\Charlie.Ikosi\OneDrive - PDP\Documents\floodsense\test\roi\Aoraki_Snow_Test.shp" # Aoraki Snow Test
DATE_RANGE = "2026-01-01/2026-01-31"
ORBIT_STATE = "ascending"            # Choose ascending or descending for orbit path of satellite

# Search all scenes
my_items, my_aoi = get_rtc_catalog_items(AOI, DATE_RANGE)

In [ ]:
# Filter scenes by orbit type
itms = select_scenes_by_orbit(my_items,orbit_state=ORBIT_STATE)
itms